# Day 1: Python, Visualization, Probability, Regression, and Deep Learning Basics

**Advanced Pathway Project 1: Modeling and Inference for Classroom Data**

Today has two goals:

1. Build enough Python confidence to read, modify, and discuss code.
2. Connect Python tools to examples that high school math, statistics, and science teachers can reuse or adapt.

**Teaching rhythm:** most sections are designed as a 10–12 minute instructor demo followed by a 5–10 minute participant task. For mixed coding backgrounds, each exercise has a **Core** task and a **Stretch** task.

## 0. Setup: Packages We Will Use

We will use four families of packages today.

| Package | Main use today | Classroom connection |
|---|---|---|
| `numpy` | arrays and numerical calculations | simulations, probability, functions |
| `pandas` | tables and summaries | grade books, lab data, survey data |
| `matplotlib` | visualization | quick classroom plots |
| `scipy.stats` | probability functions and statistical distributions | binomial, normal, sampling ideas |
| `scikit-learn` | standard regression | quick modeling workflow |
| `torch` | deep learning package | regression with trainable parameters |

If a package is missing on a workshop computer, ask for help rather than spending too much time installing during the session.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy import stats

# scikit-learn is useful for standard regression.
try:
    from sklearn.linear_model import LinearRegression
    from sklearn.metrics import mean_squared_error, r2_score
    HAS_SKLEARN = True
except Exception as e:
    HAS_SKLEARN = False
    print("scikit-learn is not available. We will use a NumPy fallback.")

# PyTorch is a common deep learning package.
try:
    import torch
    HAS_TORCH = True
except Exception as e:
    HAS_TORCH = False
    print("PyTorch is not available. You can still read the code and run the non-deep-learning examples.")

rng = np.random.default_rng(seed=42)

## 1. Python as a Modeling Notebook

A notebook lets us mix explanation, code, graphs, and reflection. In this workshop, we will use Python to answer questions such as:

- How does a quantity change over time?
- How can we visualize a dataset clearly?
- How can we use probability functions to describe uncertainty?
- How can we fit a regression model to data?
- How is a deep learning package used to fit a simple regression model?

## 2. Variables, Expressions, and Types

A variable stores a value so we can reuse it. In modeling, variables often represent parameters, initial conditions, or measurements.

In [ ]:
population = 1000
initial_infected = 5
initial_susceptible = population - initial_infected
initial_recovered = 0

print("Susceptible:", initial_susceptible)
print("Infected:", initial_infected)
print("Recovered:", initial_recovered)
print("Total:", initial_susceptible + initial_infected + initial_recovered)

### Exercise 1: Read and Modify Parameters

**Core:** Change `initial_infected` from 5 to 25. What happens to `initial_susceptible`?

**Stretch:** Create a new variable called `initial_vaccinated` and decide how it should change the susceptible count.

**Classroom adaptation:** This is like changing assumptions in a word problem before solving it.

In [ ]:
# Exercise 1 workspace
population = 1000
initial_infected = 5
initial_susceptible = population - initial_infected
initial_recovered = 0

# Modify the variables above and rerun this cell.
print(initial_susceptible, initial_infected, initial_recovered)

## 3. Functions for Model Rules

A function gives a name to a reusable calculation. Day 2 uses functions for differential equation rules, and Day 3 uses functions for loss functions and inference.

In [ ]:
def new_infections(S, I, beta, population):
    """Expected new infections in one time step."""
    return beta * S * I / population


def sir_one_step(S, I, R, beta, gamma, population, dt=1.0):
    """Advance a simple SIR model by one Euler step."""
    infections = beta * S * I / population
    recoveries = gamma * I
    S_next = S - dt * infections
    I_next = I + dt * infections - dt * recoveries
    R_next = R + dt * recoveries
    return S_next, I_next, R_next

S1, I1, R1 = sir_one_step(995, 5, 0, beta=0.30, gamma=0.10, population=1000)
print(S1, I1, R1)
print("Total:", S1 + I1 + R1)

### Exercise 2: Function Checkpoint

The SIR step has two flows: susceptible people move into infected, and infected people move into recovered.

**Core:** Change `beta` to 0.10, 0.30, and 0.50. What changes?

**Stretch:** Change `gamma`. What classroom language would you use to explain this parameter?

In [ ]:
# Exercise 2 workspace
for beta_value in [0.10, 0.30, 0.50]:
    S1, I1, R1 = sir_one_step(995, 5, 0, beta=beta_value, gamma=0.10, population=1000)
    print("beta =", beta_value, "next infected =", round(I1, 2))

## 4. Lists, NumPy Arrays, and First Plots

A list stores several values. A NumPy array is better for numerical work because it supports vectorized mathematical operations and plotting.

In [ ]:
time = np.arange(0, 11, 1)
infected = np.array([5, 6, 7, 9, 12, 16, 20, 24, 28, 31, 33])

plt.figure(figsize=(7, 4))
plt.plot(time, infected, marker="o")
plt.title("Example Infection Counts")
plt.xlabel("Day")
plt.ylabel("Number infected")
plt.grid(True, alpha=0.3)
plt.show()

### Exercise 3: Plotting Checkpoint

**Core:** Add a second array called `recovered` and plot it on the same graph.

**Stretch:** Add a legend and change the title to something your students would understand.

In [ ]:
# Exercise 3 workspace
time = np.arange(0, 11, 1)
infected = np.array([5, 6, 7, 9, 12, 16, 20, 24, 28, 31, 33])
recovered = np.array([0, 0, 1, 1, 2, 3, 5, 8, 11, 15, 18])

plt.figure(figsize=(7, 4))
plt.plot(time, infected, marker="o", label="Infected")
plt.plot(time, recovered, marker="s", label="Recovered")
plt.xlabel("Day")
plt.ylabel("Number of people")
plt.title("Two Compartments Over Time")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 5. Data Visualization with Classroom-Style Data

Participants are especially interested in visualization, so we will use a small dataset that resembles what teachers might see: study time, quiz scores, lab group, and absences.

Good classroom visualizations usually answer one question at a time:

- **Line plot:** How does something change over time?
- **Bar chart:** How do groups compare?
- **Histogram:** What is the distribution?
- **Scatter plot:** How are two numerical variables related?
- **Box plot:** How do distributions compare across groups?
- **Heat map/correlation table:** Which numerical variables move together?

In [ ]:
rng = np.random.default_rng(seed=7)
n_students = 36

class_data = pd.DataFrame({
    "student_id": np.arange(1, n_students + 1),
    "study_minutes": rng.integers(20, 180, size=n_students),
    "absences": rng.integers(0, 6, size=n_students),
    "lab_group": rng.choice(["A", "B", "C"], size=n_students),
})

# Create a realistic score: study helps, absences hurt, and there is random variation.
class_data["quiz_score"] = (
    55
    + 0.18 * class_data["study_minutes"]
    - 2.5 * class_data["absences"]
    + rng.normal(0, 7, size=n_students)
).clip(0, 100).round(1)

class_data.head()

In [ ]:
# Visualization example 1: distribution of quiz scores
plt.figure(figsize=(7, 4))
plt.hist(class_data["quiz_score"], bins=8, edgecolor="black")
plt.title("Distribution of Quiz Scores")
plt.xlabel("Quiz score")
plt.ylabel("Number of students")
plt.show()

# Visualization example 2: relationship between study time and score
plt.figure(figsize=(7, 4))
plt.scatter(class_data["study_minutes"], class_data["quiz_score"])
plt.title("Study Time vs. Quiz Score")
plt.xlabel("Study time in minutes")
plt.ylabel("Quiz score")
plt.grid(True, alpha=0.3)
plt.show()

# Visualization example 3: compare groups
mean_by_group = class_data.groupby("lab_group")["quiz_score"].mean()
plt.figure(figsize=(6, 4))
plt.bar(mean_by_group.index, mean_by_group.values)
plt.title("Average Quiz Score by Lab Group")
plt.xlabel("Lab group")
plt.ylabel("Average score")
plt.ylim(0, 100)
plt.show()

In [ ]:
# Visualization example 4: box plot by group
fig, ax = plt.subplots(figsize=(6, 4))
class_data.boxplot(column="quiz_score", by="lab_group", ax=ax)
plt.title("Quiz Score Distribution by Lab Group")
plt.suptitle("")
plt.xlabel("Lab group")
plt.ylabel("Quiz score")
plt.show()

# Visualization example 5: simple correlation heat map
numeric_cols = ["study_minutes", "absences", "quiz_score"]
corr = class_data[numeric_cols].corr()

plt.figure(figsize=(5, 4))
plt.imshow(corr, vmin=-1, vmax=1)
plt.colorbar(label="Correlation")
plt.xticks(range(len(numeric_cols)), numeric_cols, rotation=45, ha="right")
plt.yticks(range(len(numeric_cols)), numeric_cols)
plt.title("Correlation Heat Map")
for i in range(len(numeric_cols)):
    for j in range(len(numeric_cols)):
        plt.text(j, i, f"{corr.iloc[i, j]:.2f}", ha="center", va="center")
plt.tight_layout()
plt.show()

### Exercise 4: Visualization Choice Board

Choose one question and make a graph that answers it.

**Core choices:**

1. Do students with fewer absences tend to have higher quiz scores?
2. Which lab group has the highest median score?
3. What does the distribution of study time look like?

**Stretch choices:**

1. Create two different graphs for the same question. Which is clearer?
2. Add a short written interpretation under the graph.
3. Replace this dataset with a dataset from your own class or science lab.

In [ ]:
# Exercise 4 workspace
# Example starter: absences vs. quiz score
plt.figure(figsize=(7, 4))
plt.scatter(class_data["absences"], class_data["quiz_score"])
plt.xlabel("Absences")
plt.ylabel("Quiz score")
plt.title("Absences vs. Quiz Score")
plt.grid(True, alpha=0.3)
plt.show()

## 6. Loops for Simulation

A simulation repeats a rule many times. The current state becomes the starting point for the next step.

In [ ]:
def simulate_sir(days, S0, I0, R0, beta, gamma, population, dt=1.0):
    steps = int(days / dt)
    t = np.arange(steps + 1) * dt
    states = np.zeros((steps + 1, 3))
    states[0] = np.array([S0, I0, R0])

    for k in range(steps):
        states[k + 1] = sir_one_step(*states[k], beta, gamma, population, dt=dt)

    return t, states


t, states = simulate_sir(60, 995, 5, 0, beta=0.30, gamma=0.10, population=1000)

plt.figure(figsize=(8, 4))
plt.plot(t, states[:, 0], label="Susceptible")
plt.plot(t, states[:, 1], label="Infected")
plt.plot(t, states[:, 2], label="Recovered")
plt.xlabel("Day")
plt.ylabel("Number of people")
plt.title("Simple SIR Simulation")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

### Exercise 5: Simulation Lab

**Core:** Try `beta = 0.20`, `0.30`, and `0.45`. Which produces the largest peak infected count?

**Stretch:** Make a small table with `beta`, `peak_infected`, and `day_of_peak`.

In [ ]:
# Exercise 5 workspace
results = []
for beta_value in [0.20, 0.30, 0.45]:
    t, states = simulate_sir(60, 995, 5, 0, beta_value, 0.10, 1000)
    peak_infected = states[:, 1].max()
    day_of_peak = t[states[:, 1].argmax()]
    results.append({"beta": beta_value, "peak_infected": peak_infected, "day_of_peak": day_of_peak})

pd.DataFrame(results)

## 7. Randomness, Probability Functions, and `scipy.stats`

A probability model lets us describe uncertainty. The `scipy.stats` package gives us probability functions for many distributions.

Common functions:

| Function | Meaning | Example question |
|---|---|---|
| `.pmf(k)` | probability mass at an exact value | Probability exactly 6 students are absent |
| `.cdf(k)` | probability of being at or below a value | Probability 6 or fewer are absent |
| `.pdf(x)` | density for continuous variables | Shape of a normal curve |
| `.ppf(q)` | percentile / quantile | Score cutoff for top 10% |
| `.rvs(size=...)` | random samples | Simulate many class days |

In [ ]:
# Example: number of students who submit an assignment on time.
# Suppose each student independently submits on time with probability 0.85.
n = 30
p = 0.85
X = stats.binom(n=n, p=p)

print("P(exactly 25 submit on time):", X.pmf(25))
print("P(25 or fewer submit on time):", X.cdf(25))
print("P(at least 25 submit on time):", 1 - X.cdf(24))

k_values = np.arange(0, n + 1)
plt.figure(figsize=(8, 4))
plt.bar(k_values, X.pmf(k_values))
plt.title("Binomial Distribution: On-Time Assignments")
plt.xlabel("Number of on-time submissions")
plt.ylabel("Probability")
plt.show()

In [ ]:
# Example: normal distribution for measurement error in a science lab.
mu = 100
sigma = 5
Y = stats.norm(loc=mu, scale=sigma)

print("P(measurement is below 95):", Y.cdf(95))
print("90th percentile:", Y.ppf(0.90))

x = np.linspace(80, 120, 300)
plt.figure(figsize=(8, 4))
plt.plot(x, Y.pdf(x))
plt.axvline(Y.ppf(0.90), linestyle="--", label="90th percentile")
plt.title("Normal Model for Lab Measurement")
plt.xlabel("Measured value")
plt.ylabel("Density")
plt.legend()
plt.show()

### Exercise 6: Probability Functions

**Core:** In the binomial example, change `p` from 0.85 to 0.70. How does the graph change?

**Stretch:** Make a probability question about your own classroom context, then answer it using `.pmf`, `.cdf`, or `.ppf`.

**Teaching note:** Ask students to predict the graph before running the code.

In [ ]:
# Exercise 6 workspace
n = 30
p = 0.70
X = stats.binom(n=n, p=p)

print("P(at least 25 submit on time):", 1 - X.cdf(24))

k_values = np.arange(0, n + 1)
plt.figure(figsize=(8, 4))
plt.bar(k_values, X.pmf(k_values))
plt.title("Try a Different On-Time Probability")
plt.xlabel("Number of on-time submissions")
plt.ylabel("Probability")
plt.show()

## 8. Regression Analysis with a Statistics / Machine Learning Package

Regression asks how one or more input variables are related to an output variable.

Example classroom question:

> Can study time help predict quiz score?

This is a teaching-friendly example because participants can discuss scatter plots, slope, intercept, residuals, and limitations.

In [ ]:
X = class_data[["study_minutes"]].to_numpy()
y = class_data["quiz_score"].to_numpy()

if HAS_SKLEARN:
    reg = LinearRegression()
    reg.fit(X, y)
    y_pred = reg.predict(X)
    slope = reg.coef_[0]
    intercept = reg.intercept_
    r2 = r2_score(y, y_pred)
else:
    slope, intercept = np.polyfit(class_data["study_minutes"], y, deg=1)
    y_pred = slope * class_data["study_minutes"].to_numpy() + intercept
    r2 = 1 - np.sum((y - y_pred) ** 2) / np.sum((y - y.mean()) ** 2)

print(f"Predicted score = {intercept:.2f} + {slope:.3f} * study_minutes")
print(f"R-squared = {r2:.3f}")

plt.figure(figsize=(7, 4))
plt.scatter(class_data["study_minutes"], y, label="Observed")
plt.plot(class_data["study_minutes"], y_pred, label="Regression line")
plt.xlabel("Study time in minutes")
plt.ylabel("Quiz score")
plt.title("Linear Regression: Study Time Predicting Quiz Score")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# Residual plot: what did the model miss?
residuals = y - y_pred

plt.figure(figsize=(7, 4))
plt.scatter(y_pred, residuals)
plt.axhline(0, linestyle="--")
plt.xlabel("Predicted score")
plt.ylabel("Residual = observed - predicted")
plt.title("Residual Plot")
plt.grid(True, alpha=0.3)
plt.show()

### Exercise 7: Regression Interpretation

**Core:** Write one sentence interpreting the slope. Avoid saying the model proves causation.

**Stretch:** Fit a model with both `study_minutes` and `absences` as input variables.

**Classroom discussion:** What variables are missing? Sleep? Prior knowledge? Test anxiety? Internet access?

In [ ]:
# Exercise 7 workspace: two-variable regression
X2 = class_data[["study_minutes", "absences"]].to_numpy()
y = class_data["quiz_score"].to_numpy()

if HAS_SKLEARN:
    reg2 = LinearRegression().fit(X2, y)
    y_pred2 = reg2.predict(X2)
    print("Intercept:", reg2.intercept_)
    print("Coefficients for [study_minutes, absences]:", reg2.coef_)
    print("R-squared:", r2_score(y, y_pred2))
else:
    # NumPy least squares fallback
    X_design = np.column_stack([np.ones(len(X2)), X2])
    coef, *_ = np.linalg.lstsq(X_design, y, rcond=None)
    y_pred2 = X_design @ coef
    print("[Intercept, study_minutes coefficient, absences coefficient]:", coef)

## 9. Deep Learning Package: Regression with PyTorch

Deep learning packages are built around **tensors**, **parameters**, **loss functions**, and **optimization**.

For a first example, we will use PyTorch to fit the same linear regression model:

$$
\widehat{y} = w x + b.
$$

This is not a deep neural network yet, but it uses the same training loop structure that neural networks use.

In [ ]:
if HAS_TORCH:
    torch.manual_seed(42)

    # Scale study minutes to make training easier.
    x_np = class_data[["study_minutes"]].to_numpy(dtype="float32")
    y_np = class_data[["quiz_score"]].to_numpy(dtype="float32")
    x_scaled = (x_np - x_np.mean()) / x_np.std()

    x_tensor = torch.tensor(x_scaled)
    y_tensor = torch.tensor(y_np)

    model = torch.nn.Linear(1, 1)
    loss_fn = torch.nn.MSELoss()
    optimizer = torch.optim.SGD(model.parameters(), lr=0.05)

    losses = []
    for epoch in range(300):
        optimizer.zero_grad()
        pred = model(x_tensor)
        loss = loss_fn(pred, y_tensor)
        loss.backward()
        optimizer.step()
        losses.append(loss.item())

    print("Final training loss:", losses[-1])
    print("Learned weight and bias:", list(model.parameters()))
else:
    print("PyTorch is not installed in this environment.")

In [ ]:
if HAS_TORCH:
    with torch.no_grad():
        torch_pred = model(x_tensor).numpy().ravel()

    plt.figure(figsize=(7, 4))
    plt.plot(losses)
    plt.xlabel("Epoch")
    plt.ylabel("Mean squared error")
    plt.title("PyTorch Training Loss")
    plt.grid(True, alpha=0.3)
    plt.show()

    plt.figure(figsize=(7, 4))
    plt.scatter(class_data["study_minutes"], y_np.ravel(), label="Observed")
    plt.scatter(class_data["study_minutes"], torch_pred, label="PyTorch predictions")
    plt.xlabel("Study time in minutes")
    plt.ylabel("Quiz score")
    plt.title("PyTorch Regression Predictions")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()

### Exercise 8: Deep Learning Training Loop

**Core:** Change the learning rate from `0.05` to `0.005` or `0.5`. What happens to the loss curve?

**Stretch:** Change the number of epochs. How many epochs are enough for this small dataset?

**Discussion question:** Why is this called “training” even though the model is only a line?

In [ ]:
# Exercise 8 workspace
# Copy the PyTorch training cell above and change lr or epochs.

## 10. Optional Stretch: A Small Neural Network for a Curved Pattern

A neural network becomes useful when the relationship is not well described by a straight line.

This example uses a synthetic science-style dataset: temperature and reaction rate with a curved relationship.

In [ ]:
rng = np.random.default_rng(seed=11)
temp = np.linspace(0, 100, 80)
rate = 20 + 0.8 * temp - 0.006 * temp**2 + rng.normal(0, 3, size=len(temp))

plt.figure(figsize=(7, 4))
plt.scatter(temp, rate)
plt.xlabel("Temperature")
plt.ylabel("Reaction rate")
plt.title("Curved Science Dataset")
plt.grid(True, alpha=0.3)
plt.show()

if HAS_TORCH:
    x = torch.tensor(((temp - temp.mean()) / temp.std()).reshape(-1, 1), dtype=torch.float32)
    y = torch.tensor(rate.reshape(-1, 1), dtype=torch.float32)

    nn_model = torch.nn.Sequential(
        torch.nn.Linear(1, 8),
        torch.nn.ReLU(),
        torch.nn.Linear(8, 1)
    )
    optimizer = torch.optim.Adam(nn_model.parameters(), lr=0.02)
    loss_fn = torch.nn.MSELoss()

    for epoch in range(1000):
        optimizer.zero_grad()
        pred = nn_model(x)
        loss = loss_fn(pred, y)
        loss.backward()
        optimizer.step()

    with torch.no_grad():
        nn_pred = nn_model(x).numpy().ravel()

    plt.figure(figsize=(7, 4))
    plt.scatter(temp, rate, label="Observed")
    plt.plot(temp, nn_pred, label="Neural network fit")
    plt.xlabel("Temperature")
    plt.ylabel("Reaction rate")
    plt.title("Small Neural Network Regression")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()

## 11. From Model Output to Data: Inference Preview

Day 2 and Day 3 connect modeling and data fitting. The basic idea is:

1. Choose parameter values.
2. Simulate a model.
3. Compare model output with data using a loss function.
4. Search for parameter values with smaller loss.

In [ ]:
t, true_states = simulate_sir(40, 995, 5, 0, beta=0.30, gamma=0.10, population=1000)
true_infected = true_states[:, 1]

rng = np.random.default_rng(seed=123)
observed_infected = rng.poisson(np.maximum(true_infected, 0))

plt.figure(figsize=(8, 4))
plt.plot(t, true_infected, label="Model infected")
plt.scatter(t, observed_infected, s=15, label="Noisy observations")
plt.xlabel("Day")
plt.ylabel("Infected")
plt.title("Model Output and Noisy Data")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
def simulate_infected_curve(beta, gamma=0.10):
    t, states = simulate_sir(40, 995, 5, 0, beta=beta, gamma=gamma, population=1000)
    return states[:, 1]


def squared_error_score(beta, observed):
    model_infected = simulate_infected_curve(beta)
    return np.sum((observed - model_infected) ** 2)

beta_grid = np.linspace(0.10, 0.50, 81)
scores = np.array([squared_error_score(beta_value, observed_infected) for beta_value in beta_grid])
best_beta = beta_grid[np.argmin(scores)]

plt.figure(figsize=(8, 4))
plt.plot(beta_grid, scores)
plt.axvline(best_beta, linestyle="--", label=f"best beta = {best_beta:.2f}")
plt.xlabel("beta")
plt.ylabel("Squared error")
plt.title("Simple Grid Search for beta")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

### Exercise 9: End-of-Day Synthesis

Work with a partner.

1. Which visualization from today would you most likely use in your class?
2. Which regression example felt most teachable?
3. What is one place where students might confuse correlation, prediction, and causation?
4. What would help students with limited coding experience participate?

## Day 2 Preview

Tomorrow we will move from Python foundations to dynamical systems: discrete updates, continuous-time models, Euler simulation, SIS/SIR/SIRH ideas, loss functions, and optimization.